# Synthesis

What does the whole study say? Group A describes what decision models are like (outputs, stability, geometry); Group B asks what their probabilities add to tabular predictors compared with frozen embeddings, label-trained alternatives and native-text systems; Group C looks inside Strands and trains it on labels. This notebook brings the groups together without redrawing their panels: one study-at-a-glance figure, the confirmatory family (statistical framework approved by the owner on 24-11-2026), and the status of the planned paper figures. Datasets are the unit (18: 9 classification, 9 regression); raw gains stay within a task family and only the headroom-normalised gain and the MulTaBench-style normalised score pool all 18.

In [1]:
import sys
from pathlib import Path
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file() and (p / "src").is_dir())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
import matplotlib.pyplot as plt
import pandas as pd
from src.utils.notebook_display import display_local, display_json, display_table
from src.utils.notebook_report import begin_report, report_section, record, finish_report
from src.utils.provenance import provenance
from src.visualize import figures, style
style.apply()

from src.evaluation import learner_scores, study_audit
from src.evaluation.synthesis_overview import glance_table
from src.visualize.overview import confirmatory_forest, study_at_a_glance
NOTEBOOK = "general/5_synthesis"
begin_report()
save = figures.FigureSaver(NOTEBOOK)
report_section("1. Scope and provenance")
display_json("Notebook execution identity", provenance())
exports = study_audit.export_status()
frames = exports.attrs["frames"]
display_table("Result exports read by this synthesis", exports[["study", "notebook", "status", "created_utc"]], preview_rows=12)

study,notebook,status,created_utc
A1,A/exp01_outputs,superseded_schema,2026-11-18T13:24:39.664228+00:00
A2,A/exp02_robustness,awaiting_export,None
A3,A/exp03_geometry,awaiting_export,None
B1,B/exp04_decision_features,awaiting_export,None
B2,B/exp05_frozen_embeddings,awaiting_export,None
B3,B/exp08_when_text_helps,awaiting_export,None
B4,B/exp09_data_efficiency,awaiting_export,None
B5,B/exp07_native_text,awaiting_export,None
C1,C/exp10_representations,awaiting_export,None
C2,C/exp11_adaptation,awaiting_export,None


## Study at a glance

For each prespecified arm, the gain over the same predictor without text: mean and median with dataset-bootstrap intervals; pooled headroom-normalised gain, the pooled MulTaBench-style normalised score (once the fold tables exist: per dataset and fold, min-max over every predictor × arm shown plus no text, negative R² clipped at 0, averaged over predictors; relative to that set) and raw ΔAUC / ΔR² per task family in separate panels. Decision rows use the median over all 17 decision models; the Strands row is the prespecified LoRA + head cell (no winner is selected).

In [2]:
report_section("2. Study at a glance")
glance = glance_table({s: frames.get(s) or {} for s in ("B1", "B2", "C2")})
if glance.empty:
    print("Waiting for export: the B1, B2 and C2 exports are not held locally.")
else:
    display_table("Study at a glance", glance.round(4), preview_rows=20)
    fig, name, caption = study_at_a_glance(glance)
    save(fig, name, caption=caption)
    display_local(fig)
    plt.close(fig)

Waiting for export: the B1, B2 and C2 exports are not held locally.


## Group A: what decision models are like

A1 output shape, confidence and leave-one-out consensus; A2 change under equivalent presentations relative to replay; A3 geometry against frozen encoders. The model map and profile table live in `A/A_synthesis`.

In [3]:
report_section("3. Group A")
for study in ("A1", "A2", "A3"):
    row = exports.set_index("study").loc[study]
    print(f"{study} ({row.notebook}): {row.status}" + (f" — expected {row.note}" if row.note else ""))

A1 (A/exp01_outputs): superseded_schema — expected jevpfn.rq1a.report.v2
A2 (A/exp02_robustness): awaiting_export
A3 (A/exp03_geometry): awaiting_export


## Groups B and C: what the probabilities add, and Strands from the inside

Per-model gains, ranks and fold-level win rates (B1), representation comparisons (B2, B5), when text helps (B3), amount of labelled data (B4), hidden states (C1) and label training (C2) are drawn in their own notebooks and in `B/B_synthesis` and `C/C_synthesis`.

In [4]:
report_section("4. Groups B and C")
for study in ("B1", "B2", "B3", "B4", "B5", "C1", "C2"):
    row = exports.set_index("study").loc[study]
    print(f"{study} ({row.notebook}): {row.status}")

B1 (B/exp04_decision_features): awaiting_export
B2 (B/exp05_frozen_embeddings): awaiting_export
B3 (B/exp08_when_text_helps): awaiting_export
B4 (B/exp09_data_efficiency): awaiting_export
B5 (B/exp07_native_text): awaiting_export
C1 (C/exp10_representations): awaiting_export
C2 (C/exp11_adaptation): awaiting_export


## Statistical framework and confirmatory family

The declared confirmatory family (`config/05_synthesis/analysis.yaml`): two-sided Wilcoxon signed-rank tests with Holm over the whole family, Bayesian signed-rank posteriors with a ROPE; Friedman/CD, fold-level win rates and the normalised score are descriptive. Approved by the owner on 24-11-2026: 21 pooled contrasts (decision features vs no text, vs frozen E5-small and vs TAR E5-small, each per predictor, on the headroom-normalised gain of all 18 datasets). The family runs only when every export it needs exists; until then this section prints the waiting state.

In [5]:
report_section("5. Statistical framework")
state = learner_scores.framework_state()
family = learner_scores.confirmatory_family_state()
print(f"framework.status = {state['framework_status']}; approved: {state['approved']}")
print(f"Confirmatory family: {family['status']}; {family['declared_contrasts']} declared contrasts "
      f"({family['pooled_contrasts']} pooled, {family['per_task_contrasts']} per task).")
display_table("Planned learner comparisons per experiment", learner_scores.applications_table(), preview_rows=12)
display_json("Confirmatory family state", family)
confirmatory = learner_scores.run_confirmatory_family()
print(f"Confirmatory family run: {confirmatory['status']}"
      + (f"; waiting for {', '.join(confirmatory['missing'])}" if confirmatory.get("missing") else "")
      + (f"; {confirmatory['reason']}" if confirmatory.get("reason") else ""))
if confirmatory["status"] == "compared":
    display_table("Confirmatory family: two-sided Wilcoxon, Holm over every declared contrast, Bayesian signed-rank", confirmatory["results"], preview_rows=21)
    fig, name, caption = confirmatory_forest(confirmatory["forest"])
    save(fig, name, caption=caption)
    display_local(fig)
    plt.close(fig)

framework.status = approved; approved: True
Confirmatory family: ready_to_run; 21 declared contrasts (21 pooled, 0 per task).


application,comparison,procedure,inference,score,export,gate
A1_outputs,17_decision_models,friedman_ranks_cd_diagram,descriptive,log_loss,A1,approved
A2_stability,averaged_presentations_vs_canonical_per_model,wilcoxon_holm_over_17_models_plus_bayesian,confirmatory,proper_score_skill,A2,approved
A3_geometry,None,None,descriptive_only,None,None,approved
B1_decision_features,decision_modes_vs_non_text_control_per_predictor,wilcoxon_holm_plus_bayesian,confirmatory,headroom_normalised_gain,B1,approved
B2_representations,decision_vs_frozen_embeddings_tar_label_classifier,wilcoxon_holm_plus_bayesian,confirmatory,headroom_normalised_gain,B2,approved
B3_when_text_helps,three_lodo_rules_vs_always_and_never,wilcoxon_on_regret_plus_bayesian,exploratory,regret,B3,approved
B4_data_efficiency,arms_vs_baseline_per_size,wilcoxon_holm_vs_control,exploratory,headroom_normalised_gain,B4,approved
B5_native_text_systems,native_text_systems,friedman_cd_diagram,descriptive,headline_metric,B5,approved
C1_strands_states,14_arms_vs_non_text_control,friedman_then_wilcoxon_holm_vs_control,exploratory,headroom_normalised_gain,C1,approved
C2_strands_adaptation,2x2_zero_shot_head_lora_both,friedman_then_all_pairwise_wilcoxon_holm,exploratory,headroom_normalised_gain,C2,approved


Confirmatory family run: awaiting_export; waiting for B1_decision_features, B2_representations


## Paper figures

Planned paper figures (`config/05_synthesis/paper_figures.yaml`) and which saved figures already exist in their owning notebooks. Placement stays undecided until the owner decides.

In [6]:
report_section("6. Paper figures")
from src.utils.paper_figures import resolve
display_table("Paper figure registry", resolve(), preview_rows=20)

id,title,notebook,kind,placement,planned_figures,saved_figures,status,note
fig1_headline_gain,What decision models add - per-dataset gains and model ranks,B/exp04_decision_features,figure,undecided,"gain_landscape, model_rank_heatmap",pending,planned,
fig2_representation_ladder_and_compute,"Decision features against embedding, label-trained and native alternatives, and compute",B/B_synthesis,figure,undecided,"representation_ladder*, quality_vs_compute*",pending,planned,
fig3_dataset_and_model_explanations,When do decision features help - dataset moderators and decision models,B/exp08_when_text_helps,figure,undecided,"raw_gain_vs_headroom*, model_diagnostics_vs_usefulness*",pending,planned,
fig4_fields_and_prediction_in_advance,"Which text columns help, and predicting usefulness in advance",B/exp08_when_text_helps,figure,undecided,"column_refits*, lodo*",pending,planned,
fig_study_at_a_glance,Study at a glance - every prespecified arm against no text,general/5_synthesis,figure,undecided,study_at_a_glance,pending,available_when_results_exist,
exp09_representations_panel,"Inside the decision model (Strands hidden states, Group C1)",C/exp10_representations,figure,undecided,"c1_position_path, c1_contrasts, c1_vector_vs_probabilities_by_dataset, c1_gains_per_column, c1_gains_joint",pending,available_when_results_exist,
tab_rq3_adaptation,Learning from labels with Strands (Group C2),C/exp11_adaptation,table,undecided,"c2_factorial, c2_cells, c2_context, c2_datasets, c2_gains",pending,available_when_results_exist,
fig_strands_ladder,Strands from zero-shot probabilities to label training,C/C_synthesis,figure,undecided,strands_ladder,pending,available_when_results_exist,


## Complete report

In [7]:
finish_report(NOTEBOOK, save.summary() + "\nOnly verified exports and configuration were read; no inference, model training or live VSC changes were performed.")

general/5_synthesis

1. Scope and provenance


Notebook execution identity
{
  "code_sha256": "ae47342a8c0ecd61ac29f160dd195690d8ecfe067a26ce4bb59cd7091e41ba9d",
  "git_head": "c0eae19b07a79d71cf6ea82f82bd3995d127dd37",
  "git_dirty": true,
  "tfm_library_pin": "4574eea0c2ce6646b5d2e9c7f53a6ef3882c368c dm-library (remotes/origin/HEAD)\n 248f137046487e0d57360abba1e060420ae54e1a tfm-library (remotes/origin/HEAD)\n 2208cf68d6351eff2cb0ddb5a9eb1d822523457b vsc-library (heads/main-8-g2208cf6)",
  "environment": {
    "python": "3.12.10",
    "numpy": "2.5.4",
    "pandas": "2.3.3",
    "PyYAML": "6.0.3",
    "matplotlib": "3.11.2",
    "pyarrow": "25.0.1"
  }
}


Result exports read by this synthesis
study                  notebook            status                      created_utc
   A1           A/exp01_outputs superseded_schema 2026-11-18T13:24:39.664228+00:00
   A2        A/exp02_robustness   awaiting_export                             None
   A3          A/exp03_geometry   awaiting_exp